# Calculate the spatial gradient of a field on the mesh

**Step 1: Create FELiCSMesh, FELiCSSpace and Field**

This should be familiar from the last How-To. If you need a quick refresher please check [link]

In [ ]:
from FELiCS.SpaceDisc.FELiCSMesh import FELiCSMesh

meshFileName = "./square_mesh.msh"
coordinateSystem = "Cartesian"

# Create a FELiCS mesh from the gmsh file
mesh = FELiCSMesh(coordinateSystem=coordinateSystem, meshFileName=meshFileName)

from FELiCS.SpaceDisc.FEMSpaces import getFELiCSSpace

scalarSpace = getFELiCSSpace(mesh, order=2, dim=1)
vectorSpace = getFELiCSSpace(mesh, order=2, dim=2)

from FELiCS.Fields.Field import Field

scalarField = Field(scalarSpace, mesh=mesh)
scalarField.importH5("function_values_sine")
gradientFieldsol = Field(vectorSpace, mesh=mesh)
gradientFieldsol.importH5("grad_solution")

Info     | FELiCSMesh.py          | __init__                   (line 28  ) : Opening mesh file: ./square_mesh.msh
Info     | FELiCSMesh.py          | __init__                   (line 36  ) : Mesh contains 513 nodes and 1024 elements


**Step 2: Define the gradient Expression and evaluate it on the Field**
We now want to calculate the gradient of our scalar quantity using the `getGradient()` method of our Field class. This method returns a Field 

In [4]:

from ufl import TestFunction, dx, conj
from FELiCS.Misc.tensorUtils import iGrad, iConj, iDot, Tensor
import dolfinx.fem

J_hat = mesh.coordinateSystem.J_hat
print(scalarSpace.element)
v = TestFunction(vectorSpace)
v_tens = Tensor(v, CoordSys=mesh.coordinateSystem, hasSpectralDimension=True)
expression = iDot(iGrad(scalarField.getTensor()), iConj(v_tens)).ufl_tens * J_hat* dx

gradientField = scalarField.getGradientField()

print(sum(gradientField.getCoefficientArray() - gradientFieldsol.getCoefficientArray()))

(-0.1487525173112279+0j)
